# Lab 04: RAGAS Evaluation

**Goal:** Build a complete evaluation pipeline for RAG systems using RAGAS, with custom judge metrics and regression detection.

**What you'll build:**
1. A minimal RAG system to evaluate (naive retrieval + LLM generation via the configured provider)
2. A golden dataset with 10 labeled queries
3. RAGAS evaluation: Faithfulness, Answer Relevance, Context Precision, Context Recall
4. Custom LLM-as-judge metrics
5. Regression detection pattern

**Prerequisites:** `AI_PROVIDER` / `AI_MODEL` / `AI_API_KEY` set in `.env` (see `ai_client.py`).

In [ ]:
!pip install ragas langchain-google-genai langchain-anthropic langchain-openai langchain-ollama langchain-huggingface datasets faiss-cpu sentence-transformers google-genai anthropic openai python-dotenv --quiet

## 1. Build the RAG System to Evaluate

In [ ]:
import numpy as np
import faiss
import ai_client
from sentence_transformers import SentenceTransformer

CORPUS = [
    {"id": "r0", "text": "Naive RAG retrieves the top-k most similar document chunks using cosine similarity over dense embeddings, then passes them to an LLM to generate an answer. It has three stages: indexing, retrieval, and generation."},
    {"id": "r1", "text": "Advanced RAG adds pre-retrieval optimizations (query rewriting, HyDE) and post-retrieval optimizations (reranking, contextual compression) to improve over naive RAG."},
    {"id": "r2", "text": "Chunking splits documents into smaller pieces before indexing. Recursive character splitting is the recommended default. Chunk size of 512 tokens with 10-20% overlap works well for most use cases."},
    {"id": "r3", "text": "Embeddings convert text to dense vectors. Cosine similarity measures semantic relatedness. Models like BGE-M3, text-embedding-3-small, and E5-large are production-grade choices."},
    {"id": "r4", "text": "HNSW (Hierarchical Navigable Small World) is the standard ANN index for vector databases. It achieves O(log N) query time. For corpora over 100M vectors, IVF+PQ is preferred for memory efficiency."},
    {"id": "r5", "text": "Reranking is a two-stage process: retrieve 20-50 candidates with a fast bi-encoder, then rerank with a cross-encoder that scores each (query, passage) pair jointly. This is more accurate but slower."},
    {"id": "r6", "text": "Hybrid search combines BM25 (keyword) and dense retrieval. Results are merged with Reciprocal Rank Fusion (RRF). It outperforms either method alone on corpora with both exact-match and semantic queries."},
    {"id": "r7", "text": "RAGAS measures four properties: Faithfulness (is the answer grounded in context?), Answer Relevance (does the answer address the question?), Context Precision, and Context Recall."},
    {"id": "r8", "text": "Faithfulness is the most important RAGAS metric. A faithfulness score below 0.8 indicates the system is hallucinating — generating claims not supported by the retrieved context."},
    {"id": "r9", "text": "Graph RAG extracts a knowledge graph from documents. It retrieves graph subgraphs and community summaries alongside text chunks. It outperforms standard RAG on multi-hop queries."},
    {"id": "r10", "text": "Self-RAG trains a model with special tokens to decide when to retrieve, evaluate retrieved passages, and critique its own output. It requires fine-tuning on a labeled dataset."},
    {"id": "r11", "text": "HyDE (Hypothetical Document Embeddings) generates a hypothetical answer to the query, embeds that answer, and uses it for retrieval. This works because hypothetical answers are distributionally closer to real documents than raw queries."},
    {"id": "r12", "text": "Multi-tenancy in RAG requires namespace isolation (separate vector index per tenant) or metadata filtering (shared index with tenant_id filter). Namespace isolation is more secure; filtering is cheaper."},
    {"id": "r13", "text": "FLARE (Forward-Looking Active REtrieval) generates text speculatively and pauses when uncertain (low token probability), triggering a new retrieval pass. It enables on-demand retrieval during generation."},
    {"id": "r14", "text": "Context window overflow occurs when retrieved chunks exceed the LLM's context limit. Mitigations: contextual compression (extract relevant sentences), reduce k, or use a long-context model."},
]

# Build dense index
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
corpus_embs = embed_model.encode(
    [d["text"] for d in CORPUS],
    normalize_embeddings=True,
    show_progress_bar=False
)
faiss_index = faiss.IndexFlatIP(corpus_embs.shape[1])
faiss_index.add(corpus_embs.astype(np.float32))


def retrieve(query: str, k: int = 4) -> list[str]:
    q_emb = embed_model.encode(
        f"Represent this sentence for searching relevant passages: {query}",
        normalize_embeddings=True
    ).reshape(1, -1).astype(np.float32)
    _, indices = faiss_index.search(q_emb, k)
    return [CORPUS[i]["text"] for i in indices[0]]


def generate(query: str, contexts: list[str]) -> str:
    context_str = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    return ai_client.generate(
        prompt=f"Context:\n{context_str}\n\nQuestion: {query}",
        system="Answer the question using only the provided context.",
        max_tokens=256,
    )


# Quick test
q = "What is HyDE?"
ctx = retrieve(q)
ans = generate(q, ctx)
print(f"Q: {q}\nA: {ans}")

## 2. Golden Dataset (10 labeled queries)

In [ ]:
# Golden dataset: (query, expected_answer, relevant_chunk_ids)
# In production, build this from real user logs + human annotation

GOLDEN_DATASET = [
    {
        "query": "What is naive RAG and what are its three stages?",
        "ground_truth": "Naive RAG retrieves the top-k most similar chunks using cosine similarity, then passes them to an LLM. Its three stages are indexing, retrieval, and generation.",
        "relevant_ids": ["r0"],
    },
    {
        "query": "How does advanced RAG improve over naive RAG?",
        "ground_truth": "Advanced RAG adds pre-retrieval optimizations like query rewriting and HyDE, plus post-retrieval optimizations like reranking and contextual compression.",
        "relevant_ids": ["r1"],
    },
    {
        "query": "What chunk size and overlap work well for most RAG use cases?",
        "ground_truth": "A chunk size of 512 tokens with 10-20% overlap works well for most use cases, using recursive character splitting.",
        "relevant_ids": ["r2"],
    },
    {
        "query": "What ANN index should I use for 50 million vectors?",
        "ground_truth": "HNSW is the standard choice for corpora under 100M vectors. For over 100M vectors, IVF+PQ is preferred for memory efficiency.",
        "relevant_ids": ["r4"],
    },
    {
        "query": "What is two-stage retrieval with reranking?",
        "ground_truth": "Retrieve 20-50 candidates with a fast bi-encoder, then rerank with a cross-encoder that scores each query-passage pair jointly for higher accuracy.",
        "relevant_ids": ["r5"],
    },
    {
        "query": "How does RRF merge BM25 and dense results?",
        "ground_truth": "Reciprocal Rank Fusion merges ranked lists by summing 1/(k+rank) for each document. It doesn't require calibrated scores, just ranks.",
        "relevant_ids": ["r6"],
    },
    {
        "query": "What does RAGAS faithfulness measure?",
        "ground_truth": "Faithfulness measures whether the answer is grounded in the retrieved context. A score below 0.8 indicates hallucination.",
        "relevant_ids": ["r7", "r8"],
    },
    {
        "query": "What is HyDE and why does it work?",
        "ground_truth": "HyDE generates a hypothetical answer to the query and uses its embedding for retrieval. It works because hypothetical answers are distributionally closer to real documents than raw queries.",
        "relevant_ids": ["r11"],
    },
    {
        "query": "What are the two approaches to multi-tenancy in RAG?",
        "ground_truth": "Namespace isolation (separate vector index per tenant) and metadata filtering (shared index with tenant_id filter). Namespace isolation is more secure; filtering is cheaper.",
        "relevant_ids": ["r12"],
    },
    {
        "query": "What happens when retrieved chunks exceed the context window?",
        "ground_truth": "Context window overflow occurs. Mitigations include contextual compression, reducing k, or using a long-context model.",
        "relevant_ids": ["r14"],
    },
]

print(f"Golden dataset: {len(GOLDEN_DATASET)} queries")

## 3. Run the System and Collect Results

In [ ]:
# Run the RAG system on all golden queries
results = []
for sample in GOLDEN_DATASET:
    contexts = retrieve(sample["query"], k=4)
    answer   = generate(sample["query"], contexts)
    results.append({
        "query":        sample["query"],
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": sample["ground_truth"],
    })
    print(f"✓ {sample['query'][:60]}...")

print(f"\nCollected {len(results)} results")

## 4. RAGAS Evaluation

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Configure RAGAS to use the configured AI_PROVIDER as the judge LLM.
# Embeddings stay local (BGE) regardless of AI_PROVIDER, since not every
# provider (e.g. Claude) exposes an embeddings API.
judge_llm   = LangchainLLMWrapper(ai_client.get_judge_llm())
judge_embed = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

# Build HuggingFace Dataset (RAGAS required format)
ragas_data = Dataset.from_list([
    {
        "question":    r["query"],
        "answer":      r["answer"],
        "contexts":    r["contexts"],
        "ground_truth": r["ground_truth"],
        "ground_truths": [r["ground_truth"]],
    }
    for r in results
])

print("RAGAS dataset columns:", ragas_data.column_names)
print(f"Number of samples: {len(ragas_data)}")

In [ ]:
# Run RAGAS evaluation
# Note: this makes LLM API calls for each metric on each sample
# For 10 samples × 4 metrics ≈ 40-80 API calls (~$0.05 with Haiku)

print("Running RAGAS evaluation... (this takes ~1-2 minutes)")

ragas_results = evaluate(
    dataset=ragas_data,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=judge_embed,
)

print("\n=== RAGAS Results ===")
metrics_dict = dict(ragas_results)
for metric, score in metrics_dict.items():
    if isinstance(score, float):
        status = "✓" if score >= 0.7 else "✗"
        print(f"  {status} {metric:25s}: {score:.3f}")

## 5. Per-Sample Scores (Diagnose Failures)

In [ ]:
# View per-sample results to identify which queries are failing
df = ragas_results.to_pandas()
cols_to_show = ["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available_cols = [c for c in cols_to_show if c in df.columns]

print("Per-sample scores (sorted by faithfulness):")
print(df[available_cols].sort_values("faithfulness").to_string(index=False))

## 6. Custom LLM-as-Judge Metrics

RAGAS doesn't cover everything. Add custom metrics for domain-specific quality dimensions.

In [ ]:
import json

def custom_conciseness(question: str, answer: str) -> float:
    """
    Score whether the answer is appropriately concise (not padded with unnecessary text).
    1.0 = perfectly concise, 0.0 = excessively padded or repetitive.
    """
    resp = ai_client.generate(
        prompt=f"Question: {question}\nAnswer: {answer}",
        system="""Rate how concise this answer is on a scale 0.0-1.0.
1.0 = directly answers the question with no padding
0.5 = some unnecessary preamble or repetition
0.0 = excessively long, repetitive, or padded with caveats
Output JSON: {"score": 0.0-1.0, "reason": "one sentence"}""",
        max_tokens=128,
        json_mode=True,
    )
    return json.loads(resp)["score"]


def custom_technical_accuracy(question: str, answer: str, ground_truth: str) -> float:
    """
    Compare the answer's technical claims against the ground truth.
    1.0 = all claims accurate, 0.0 = contradicts ground truth.
    """
    resp = ai_client.generate(
        prompt=f"Question: {question}\nReference: {ground_truth}\nAnswer: {answer}",
        system="""Rate technical accuracy of the answer vs. the reference answer (0.0-1.0).
1.0 = all technical claims match the reference
0.5 = partially correct, minor errors
0.0 = contradicts the reference on key claims
Output JSON: {"score": 0.0-1.0}""",
        max_tokens=128,
        json_mode=True,
    )
    return json.loads(resp)["score"]


# Run custom metrics on a sample of results (to control cost)
import random
sample_size = min(5, len(results))  # evaluate 5 samples
sample = random.sample(results, sample_size)

print(f"Custom metrics on {sample_size} samples:\n")
conciseness_scores = []
accuracy_scores    = []

for r in sample:
    # Match back to golden dataset for ground_truth
    gt = next(g["ground_truth"] for g in GOLDEN_DATASET if g["query"] == r["query"])
    c_score = custom_conciseness(r["query"], r["answer"])
    a_score = custom_technical_accuracy(r["query"], r["answer"], gt)
    conciseness_scores.append(c_score)
    accuracy_scores.append(a_score)
    print(f"  Q: {r['query'][:55]}...")
    print(f"     conciseness={c_score:.2f}  accuracy={a_score:.2f}")

print(f"\nMean conciseness: {sum(conciseness_scores)/len(conciseness_scores):.3f}")
print(f"Mean accuracy:    {sum(accuracy_scores)/len(accuracy_scores):.3f}")

## 7. Regression Detection Pattern

In [ ]:
import json
from pathlib import Path

BASELINE_PATH = "ragas_baseline.json"
REGRESSION_TOLERANCE = 0.03  # 3% drop triggers a block

def save_baseline(metrics: dict, path: str = BASELINE_PATH):
    """Save current metrics as the baseline. Call manually after a validated improvement."""
    baseline = {k: v for k, v in metrics.items() if isinstance(v, float)}
    Path(path).write_text(json.dumps(baseline, indent=2))
    print(f"Baseline saved: {baseline}")


def check_regression(current_metrics: dict, path: str = BASELINE_PATH) -> list[str]:
    """Return list of regression messages. Empty list = no regression."""
    if not Path(path).exists():
        print("No baseline found — saving current metrics as baseline.")
        save_baseline(current_metrics, path)
        return []

    baseline = json.loads(Path(path).read_text())
    regressions = []

    for metric, baseline_val in baseline.items():
        current_val = current_metrics.get(metric, 0)
        drop = baseline_val - current_val
        if drop > REGRESSION_TOLERANCE:
            regressions.append(
                f"{metric}: {baseline_val:.3f} → {current_val:.3f} (drop: {drop:.3f})"
            )

    return regressions


# Save the current run as baseline (first time only)
current = {k: v for k, v in metrics_dict.items() if isinstance(v, float)}
regressions = check_regression(current)

if regressions:
    print("\n⚠️  REGRESSIONS DETECTED — would block deployment:")
    for r in regressions:
        print(f"  ✗ {r}")
else:
    print("\n✓ No regressions detected. Safe to deploy.")

## 8. Recall@k Evaluation (Retrieval-Only)

RAGAS measures end-to-end quality. For diagnosing *retrieval* specifically, use Recall@k against the golden relevant chunk IDs.

In [ ]:
def compute_recall_at_k(golden_dataset: list[dict], k: int = 4) -> dict:
    """Compute retrieval Recall@k using golden relevant chunk IDs."""
    recalls = []

    for sample in golden_dataset:
        query        = sample["query"]
        relevant_ids = set(sample["relevant_ids"])

        # Retrieve top-k
        q_emb = embed_model.encode(
            f"Represent this sentence for searching relevant passages: {query}",
            normalize_embeddings=True
        ).reshape(1, -1).astype(np.float32)
        _, indices = faiss_index.search(q_emb, k)
        retrieved_ids = {CORPUS[i]["id"] for i in indices[0]}

        # Recall = |retrieved ∩ relevant| / |relevant|
        tp = len(retrieved_ids & relevant_ids)
        recall = tp / len(relevant_ids)
        recalls.append(recall)

    return {
        f"recall_at_{k}": round(sum(recalls) / len(recalls), 3),
        "per_query":       [(s["query"][:50], round(r, 2)) for s, r in zip(golden_dataset, recalls)],
    }


for k_val in [1, 3, 5, 10]:
    result = compute_recall_at_k(GOLDEN_DATASET, k=k_val)
    print(f"Recall@{k_val}: {result[f'recall_at_{k_val}']}")

print("\nPer-query Recall@4:")
detail = compute_recall_at_k(GOLDEN_DATASET, k=4)
for query, recall in detail["per_query"]:
    status = "✓" if recall >= 1.0 else "~" if recall > 0 else "✗"
    print(f"  {status} {query}...: {recall}")

## 9. Full Evaluation Summary

In [ ]:
retrieval_metrics = compute_recall_at_k(GOLDEN_DATASET, k=4)

print("="*50)
print("FULL EVALUATION SUMMARY")
print("="*50)
print()
print("Retrieval Metrics:")
print(f"  Recall@4:          {retrieval_metrics['recall_at_4']}")
print()
print("End-to-End RAGAS Metrics:")
for metric, score in current.items():
    threshold = 0.7
    status = "✓" if score >= threshold else "✗"
    print(f"  {status} {metric:25s}: {score:.3f}")
print()
print("Custom Metrics (5-sample estimate):")
print(f"  Conciseness:       {sum(conciseness_scores)/len(conciseness_scores):.3f}")
print(f"  Technical Accuracy:{sum(accuracy_scores)/len(accuracy_scores):.3f}")
print()
print("Eval Pyramid:")
print("  Level 1 (unit) ......... chunk size/overlap tests ✓")
print(f"  Level 2 (retrieval) .... Recall@4={retrieval_metrics['recall_at_4']} {'✓' if retrieval_metrics['recall_at_4'] >= 0.8 else '✗'}")
print(f"  Level 3 (RAGAS CI) ..... faithfulness={current.get('faithfulness', 0):.3f} {'✓' if current.get('faithfulness', 0) >= 0.8 else '✗'}")
print("  Level 4 (production) ... online monitoring (not shown)")

## Key Takeaways

| What | How | When |
|---|---|---|
| **Faithfulness** | RAGAS (no gold labels needed) | Every PR |
| **Answer Relevance** | RAGAS (no gold labels needed) | Every PR |
| **Context Precision/Recall** | RAGAS (needs gold labels) | Every PR |
| **Recall@k** | Custom (gold relevant_ids needed) | Every PR |
| **Custom quality** | LLM-as-judge (sample ~20%) | Every PR, 20% sample |
| **Regression detection** | Compare vs. saved baseline | Every PR |

**Threshold guidelines:**
- Faithfulness ≥ 0.80 (critical — below this = hallucination)
- Answer Relevance ≥ 0.75
- Context Recall ≥ 0.70 (below = missing relevant chunks)
- Recall@5 ≥ 0.80 (retrieval health)

**Cost:** ~10 samples × 4 metrics × ~$0.001/call = ~$0.04 per CI run with Gemini 3.5 Flash.